# Multi-Objective Assistant Evaluation

In this notebook, we evaluate a multi-objective Yelp Customer Assistant that performs:
1. **Sentiment Classification**: Predicts the star rating (1-5).
2. **Topic Extraction**: Identifies key aspects mentioned (Food, Service, etc.).
3. **Response Generation**: Generates a polite response to the customer.

In [2]:
#imports
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import json
import pandas as pd
from tqdm import tqdm

from src.config import (
    PREDICTIONS_DIR,
    ASSISTANT_SUBSET_FILE,
    ASSISTANT_PRED_FILE,
    ASSISTANT_EVAL_FILE,
)
from src.prompts import build_assistant_prompt
from src.llm_runner import call_ollama, extract_json_block, DEFAULT_MODEL

In [3]:
#predictions folder ready
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
print("Predictions dir ready:", PREDICTIONS_DIR)

Predictions dir ready: /home/rehan/Documents/yelp-ai-assignment/outputs/predictions


In [4]:
#assistant dataset load
df = pd.read_csv(ASSISTANT_SUBSET_FILE)
print("Assistant subset shape:", df.shape)
df.head()

Assistant subset shape: (100, 5)


,text,label_raw,stars,char_length,word_length
0,"I am from Houston, TX and the only similaritie...",1,2,277,51
1,This company was a nightmare to work with. The...,0,1,1926,364
2,Every time I've been to this Denny's the servi...,0,1,220,44
3,It depends on who you get to serve you. Make ...,1,2,686,135
4,My date and I decided to try something new and...,2,3,634,104


In [5]:
# parser banao
def parse_assistant_prediction(raw_output: str):
    parsed = {
        "raw_output": raw_output,
        "json_valid": False,
        "stars_pred": None,
        "key_point_pred": None,
        "business_response_pred": None,
        "parse_error": None,
    }

    try:
        json_text = extract_json_block(raw_output)
        if json_text is None:
            parsed["parse_error"] = "No JSON object found"
            return parsed

        obj = json.loads(json_text)

        stars = obj.get("stars", None)
        key_point = obj.get("key_point", None)
        business_response = obj.get("business_response", None)

        if not isinstance(stars, int):
            parsed["parse_error"] = "stars is not an integer"
            return parsed

        if stars < 1 or stars > 5:
            parsed["parse_error"] = "stars out of range"
            return parsed

        if not isinstance(key_point, str):
            parsed["parse_error"] = "key_point is not a string"
            return parsed

        if not isinstance(business_response, str):
            parsed["parse_error"] = "business_response is not a string"
            return parsed

        parsed["json_valid"] = True
        parsed["stars_pred"] = stars
        parsed["key_point_pred"] = key_point.strip()
        parsed["business_response_pred"] = business_response.strip()
        return parsed

    except Exception as e:
        parsed["parse_error"] = str(e)
        return parsed

In [6]:
#inference function
def run_assistant_inference(input_df, model_name=DEFAULT_MODEL):
    rows = []

    for _, row in tqdm(input_df.iterrows(), total=len(input_df)):
        review_text = row["text"]
        true_stars = int(row["stars"])

        prompt = build_assistant_prompt(review_text)
        raw_output = call_ollama(prompt=prompt, model=model_name, temperature=0.0)
        parsed = parse_assistant_prediction(raw_output)

        rows.append({
            "text": review_text,
            "stars_true": true_stars,
            "prompt_type": "multi_objective_assistant",
            **parsed
        })

    return pd.DataFrame(rows)

In [7]:
#pehle 5-row smoke test
sample_df = df.head(5).copy()
sample_df[["text", "stars"]]

,text,stars
0,"I am from Houston, TX and the only similaritie...",2
1,This company was a nightmare to work with. The...,1
2,Every time I've been to this Denny's the servi...,1
3,It depends on who you get to serve you. Make ...,2
4,My date and I decided to try something new and...,3


In [8]:
#5-row assistant test
assistant_test = run_assistant_inference(sample_df, model_name=DEFAULT_MODEL)
assistant_test[[
    "stars_true",
    "stars_pred",
    "json_valid",
    "parse_error",
    "key_point_pred",
    "business_response_pred"
]]

100%|███████████████████████████████████████████████████████████| 5/5 [00:10<00:00,  2.09s/it]


,stars_true,stars_pred,json_valid,parse_error,key_point_pred,business_response_pred
0,2,2,True,None,Enchilada was terrible,Sorry to hear our enchilada didn't meet your e...
1,1,1,True,None,"Company stopped work for weeks, did unsupervis...",We apologize for the poor experience with our ...
2,1,1,True,None,Poor service,Sorry to hear that you've had a consistently d...
3,2,1,True,None,Sweaty waiter was unhygienic and unprofessional,Sorry to hear about the unpleasant experience ...
4,3,3,True,None,Uncomfortable bar stools,Thank you for your feedback. We'll look into m...


In [9]:
#quick JSON validity check
print("Valid JSON count:", assistant_test["json_valid"].sum())
print("Invalid JSON count:", (assistant_test["json_valid"] == False).sum())

Valid JSON count: 5
Invalid JSON count: 0


In [10]:
# 5 row are clear and full run
assistant_df = run_assistant_inference(df, model_name=DEFAULT_MODEL)
assistant_df.head()

100%|███████████████████████████████████████████████████████| 100/100 [03:12<00:00,  1.93s/it]


,text,stars_true,prompt_type,raw_output,json_valid,stars_pred,key_point_pred,business_response_pred,parse_error
0,"I am from Houston, TX and the only similaritie...",2,multi_objective_assistant,"```\n{\n ""stars"": 2,\n ""key_point"": ""Enchila...",True,2,Enchilada was terrible,Sorry to hear our enchilada didn't meet your e...,None
1,This company was a nightmare to work with. The...,1,multi_objective_assistant,"```\n{\n ""stars"": 1,\n ""key_point"": ""Company...",True,1,"Company stopped work for weeks, did unsupervis...",We apologize for the poor experience with our ...,None
2,Every time I've been to this Denny's the servi...,1,multi_objective_assistant,"```\n{\n ""stars"": 1,\n ""key_point"": ""Poor se...",True,1,Poor service,Sorry to hear that you've had a consistently d...,None
3,It depends on who you get to serve you. Make ...,2,multi_objective_assistant,"```\n{\n ""stars"": 1,\n ""key_point"": ""Sweaty ...",True,1,Sweaty waiter was unhygienic and unprofessional,Sorry to hear about the unpleasant experience ...,None
4,My date and I decided to try something new and...,3,multi_objective_assistant,"```\n{\n ""stars"": 3,\n ""key_point"": ""Uncomfo...",True,3,Uncomfortable bar stools,Thank you for your feedback. We'll look into m...,None


In [11]:
#save predictions
assistant_df.to_csv(ASSISTANT_PRED_FILE, index=False)
print("Saved:", ASSISTANT_PRED_FILE)

Saved: /home/rehan/Documents/yelp-ai-assignment/outputs/predictions/assistant_predictions.csv


In [12]:
#summary structured
summary = {
    "total_rows": len(assistant_df),
    "json_compliance_rate": round(assistant_df["json_valid"].mean(), 4),
    "valid_rows": int(assistant_df["json_valid"].sum()),
    "invalid_rows": int((assistant_df["json_valid"] == False).sum()),
}
summary

{'total_rows': 100,
 'json_compliance_rate': np.float64(1.0),
 'valid_rows': 100,
 'invalid_rows': 0}

In [13]:
invalid_rows = assistant_df[assistant_df["json_valid"] == False].copy()
print("Invalid rows:", len(invalid_rows))
invalid_rows[["text", "raw_output", "parse_error"]].head(10)

Invalid rows: 0


,text,raw_output,parse_error


In [14]:
valid_rows = assistant_df[assistant_df["json_valid"] == True].copy()
valid_rows[[
    "text",
    "stars_true",
    "stars_pred",
    "key_point_pred",
    "business_response_pred"
]].head(10)

,text,stars_true,stars_pred,key_point_pred,business_response_pred
0,"I am from Houston, TX and the only similaritie...",2,2,Enchilada was terrible,Sorry to hear our enchilada didn't meet your e...
1,This company was a nightmare to work with. The...,1,1,"Company stopped work for weeks, did unsupervis...",We apologize for the poor experience with our ...
2,Every time I've been to this Denny's the servi...,1,1,Poor service,Sorry to hear that you've had a consistently d...
3,It depends on who you get to serve you. Make ...,2,1,Sweaty waiter was unhygienic and unprofessional,Sorry to hear about the unpleasant experience ...
4,My date and I decided to try something new and...,3,3,Uncomfortable bar stools,Thank you for your feedback. We'll look into m...
5,Yes the pizza is the best in the burgh. But d...,5,5,Best pizza and wings,Thank you for the love! We're glad you enjoy o...
6,Well I went in here looking for gifts for othe...,4,4,The reviewer got distracted and ended up buyin...,Thank you for your honesty! We're glad you fou...
7,I don't understand why the ratings are so low ...,3,3,Inconsistent food portion sizes and inaccurate...,Thank you for your feedback. We take note of y...
8,It's just not that good. We initially told the...,1,2,Food quality was disappointing,Sorry to hear that our food didn't meet your e...
9,My husband and I stayed here over the holidays...,4,5,Luxurious amenities and exceptional service,Thank you for choosing us! We're thrilled you ...


In [15]:
#evaluation 
eval_sample = valid_rows.sample(n=min(20, len(valid_rows)), random_state=42).copy()

eval_sample = eval_sample[[
    "text",
    "stars_true",
    "stars_pred",
    "key_point_pred",
    "business_response_pred"
]]

eval_sample["faithfulness_score"] = ""
eval_sample["specificity_score"] = ""
eval_sample["politeness_score"] = ""
eval_sample["relevance_score"] = ""
eval_sample["notes"] = ""

eval_sample.head()

,text,stars_true,stars_pred,key_point_pred,business_response_pred,faithfulness_score,specificity_score,politeness_score,relevance_score,notes
83,This is the best Arby's around! The wait is a ...,4,5,Consistently good food and service despite occ...,Thank you for your kind words! We appreciate y...,,,,,
53,This is my new favorite pizza restaurant. I mu...,5,5,The pizza is delicious with a great price,Thank you for your kind words! We're thrilled ...,,,,,
70,I'm giving it 3 stars because I have hope. Sin...,3,3,The espresso machine was broken and the coffee...,Sorry to hear that our espresso machine was ou...,,,,,
45,Toyama Express is really good. I came here for...,5,5,"Excellent food, service, and value",Thank you for your kind words! We're glad you ...,,,,,
44,First time eating at this location....their se...,5,4,High-quality food and excellent service,Thank you for your kind words about our food a...,,,,,


In [16]:
eval_sample.to_csv(ASSISTANT_EVAL_FILE, index=False)
print("Saved evaluation sample:", ASSISTANT_EVAL_FILE)

Saved evaluation sample: /home/rehan/Documents/yelp-ai-assignment/outputs/predictions/assistant_evaluation_sample.csv


# Evaluation Rubric

Because there is no ground-truth dataset for key-point extraction and business response generation, this notebook uses a lightweight qualitative evaluation protocol.

Each sampled output is judged on four dimensions:

1. **Faithfulness**
   - Does the key point accurately reflect the review?
   - Does the business response stay grounded in the review?

2. **Specificity**
   - Is the extracted key point specific and informative?
   - Or is it vague and generic?

3. **Politeness**
   - Is the business response respectful and professional?

4. **Relevance**
   - Does the response actually address the complaint or compliment in the review?

A small manually reviewed sample is used to identify common strengths and weaknesses of the assistant outputs.

# Final Observations

This notebook extends the review classification task into a more practical business-facing assistant setting.

The model was asked to generate:
- a star rating
- the main complaint or compliment
- a short business response

The outputs were evaluated qualitatively because no ground-truth annotations exist for key-point extraction or response generation.

## Main observations
- The model was generally able to produce valid JSON outputs in most cases.
- Key-point extraction worked best when the review expressed one clear complaint or compliment.
- Business responses were usually polite, but sometimes too generic.
- Mixed-sentiment reviews were harder, especially when both praise and criticism appeared together.
- In some cases, the assistant captured the general sentiment correctly but missed the most important actionable detail.

## Conclusion
The local Llama model shows potential as a lightweight review-understanding assistant, but the quality of key-point extraction and response usefulness still depends on review clarity and prompt design.